# Keyword Extraction Toy Example
This notebook demonstrates the redesigned keyword extraction pipeline, stage by stage, on a compact synthetic corpus.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
from scipy import sparse as sp
from leiden_module.keyword_extraction import KeywordExtractionConfig, KeywordExtractionPipeline
ARTIFACTS = Path('toy_artifacts')
ARTIFACTS.mkdir(exist_ok=True)


In [ ]:
# Build a toy corpus (5 clusters × 4 docs)
abstracts = pd.DataFrame({
    'uid': [f'D{i:03d}' for i in range(1, 21)],
    'title': [
        'Quantum sensor advances', 'Quantum noise suppression', 'Quantum calibration toolkit', 'Entangled qubit control',
        'Perovskite solar breakthroughs', 'Photovoltaic nanostructures', 'Organic solar innovation', 'Hybrid PV reliability',
        'Battery fast charging', 'Lithium plating mitigation', 'Solid-state battery roadmap', 'Battery ageing analytics',
        'AI driven diagnostics', 'Deep learning triage', 'Clinical decision transformer', 'Explainable medical AI',
        'Autonomous vehicle safety', 'ADAS perception fusion', 'Robotic fleet coordination', 'Mobility intelligence stack'
    ],
    'abstract': [
        'We build quantum sensors with improved coherence and outline calibration routines for quantum devices.',
        'Noise suppression via squeezed light enhances quantum measurement accuracy.',
        'A software toolkit automates calibration of small quantum processors in the lab.',
        'We stabilise entangled qubit control loops using reinforcement learning.',
        'Perovskite solar cells with graded bandgaps achieve higher efficiency.',
        'Nanostructured photovoltaics trap light for enhanced conversion.',
        'Organic photovoltaics gain longevity through dopant engineering.',
        'Hybrid PV systems improve reliability with adaptive controllers.',
        'Battery packs enable fast charging through thermal-aware control.',
        'Mitigating lithium plating extends electric vehicle battery lifetime.',
        'Solid-state battery roadmap discusses interface engineering.',
        'Analytics on battery ageing informs predictive maintenance.',
        'AI driven diagnostics analyse medical images for early detection.',
        'Deep learning automates emergency triage recommendations.',
        'Transformer models support clinical decisions with explanations.',
        'Explainable medical AI increases trust in radiology workflows.',
        'Autonomous vehicles improve safety via redundancy strategies.',
        'Advanced driver assistance fuses lidar radar and vision.',
        'Robotic fleets coordinate mobility services in cities.',
        'Mobility intelligence stack models traffic for autonomy.'
    ],
    'pubyear': [2018, 2019, 2020, 2021]*5
})
membership = pd.DataFrame({
    'uid': abstracts['uid'],
    'cluster_micro': [0,0,0,0, 1,1,1,1, 2,2,2,2, 3,3,3,3, 4,4,4,4]
})
abstract_path = ARTIFACTS / 'toy_abstracts.parquet'
membership_path = ARTIFACTS / 'toy_membership.parquet'
abstracts.to_parquet(abstract_path, index=False)
membership.to_parquet(membership_path, index=False)


In [ ]:
cfg = KeywordExtractionConfig(
    abstract_path=abstract_path,
    membership_path=membership_path,
    cluster_level='cluster_micro',
    include_title=True,
    title_weight=1.0,
    min_df_unigram=1,
    max_df_unigram=1.0,
    min_df_phrase=1,
    max_df_phrase=1.0,
    phrase_min_count_per_cluster=1,
    min_cluster_doc_coverage=1,
    top_n_keywords=5,
    mmr_jaccard_lambda=0.2,
    ngram_min=2,
    ngram_max=3,
    n_jobs=1,
)
pipeline = KeywordExtractionPipeline(cfg)


## Stage 0 – Fit vectorisers

In [ ]:
pipeline._fit_vectorizers()
joblib.dump(pipeline.vec_uni, ARTIFACTS / 'vec_uni.joblib')
if pipeline.vec_phrase is not None:
    joblib.dump(pipeline.vec_phrase, ARTIFACTS / 'vec_phrase.joblib')
len(pipeline.feature_names_uni), len(pipeline.feature_names_phrase or [])


## Stage 1 – Aggregate counts & save matrices

In [ ]:
pipeline._aggregate_counts()
sp.save_npz(ARTIFACTS / 'C_uni.npz', pipeline.C_uni)
sp.save_npz(ARTIFACTS / 'DF_uni.npz', pipeline.DF_uni)
if pipeline.C_phrase is not None:
    sp.save_npz(ARTIFACTS / 'C_phrase.npz', pipeline.C_phrase)
    sp.save_npz(ARTIFACTS / 'DF_phrase.npz', pipeline.DF_phrase)
np.save(ARTIFACTS / 'cluster_doc_counts.npy', pipeline.cluster_doc_counts)
pipeline.C_uni.shape


## Stage 2 – c-TF-IDF scoring

In [ ]:
top_df = pipeline._stage_scores_and_topk()
top_df


## Stage 2.5 – Alias mapping (illustrative)

In [ ]:
alias_map = {('battery ageing analytics', 'battery aging analytics'): 'battery ageing analytics'}
alias_map


## Stage 3 – Year series

In [ ]:
term_year = pipeline._compute_year_series(top_df)
def term_year_to_long(term_year_map):
    rows = []
    for cid, term_map in term_year_map.items():
        for term, counter in term_map.items():
            for year, count in counter.items():
                rows.append({'cluster_id': cid, 'term': term, 'year': year, 'count': count})
    return pd.DataFrame(rows)
df_term_year = term_year_to_long(term_year)
df_term_year.sort_values(['cluster_id', 'term', 'year'])


## Combine & persist final results

In [ ]:
final_df = top_df.assign(
    pub_year_series=[
        dict(term_year.get(int(row.cluster_id), {}).get(str(row.term), {}))
        for row in top_df.itertuples(index=False)
    ]
)
final_df.to_parquet(ARTIFACTS / 'final_keywords.parquet', index=False)
final_df


## Top-3 keywords per cluster

In [ ]:
top3 = (
    final_df.sort_values(['cluster_id', 'score'], ascending=[True, False])
            .groupby('cluster_id', group_keys=False)
            .head(3)
)
top3
